# AI Project Risk Prediction (Multi-Output ML)

This notebook implements a **synthetic, rule-based project risk dataset** and trains **multi-output models** to predict:

- Risk Type *(classification)*
- Risk Response Strategy *(classification)*
- Risk Impact *(regression)*
- Risk Probability *(regression)*

The implementation is inspired by the paper referenced in the repository README.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score,
                             mean_squared_error, r2_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
rng = np.random.RandomState(42)   #data need to be reproduced
n = 5000


In [ ]:
df = pd.DataFrame()
df['project_id'] = np.arange(1, n+1)
df['industry'] = rng.choice(['Construction','Software','Manufacturing'], n)
df['duration_months'] = rng.randint(3, 36, n)
df['pct_complete'] = rng.randint(0, 101, n)
df['initial_budget'] = rng.randint(50_000, 5_000_000, n)
df['cost_variance'] = rng.randint(-500_000, 500_000, n)
df['scope_creep'] = rng.randint(0, 10, n)
df['tech_complexity'] = rng.randint(1, 6, n)
df['oper_efficiency'] = rng.randint(1, 6, n)
df['budget_changes'] = rng.randint(1, 6, n)
df['reg_impact'] = rng.randint(1, 6, n)
df['deviate_strategy'] = rng.randint(1, 6, n)
df['market_fluct'] = rng.randint(1, 6, n)
df['reputation_impact'] = rng.randint(1, 6, n)
df['env_impact'] = rng.randint(1, 6, n)
df['legal_challenges'] = rng.randint(1, 6, n)
df['security_threats'] = rng.randint(1, 6, n)
df['supply_disrupt'] = rng.randint(1, 6, n)
df['hr_issues'] = rng.randint(1, 6, n)
df['schedule_changes'] = rng.randint(1, 6, n)
df['schedule_variance_days'] = rng.randint(-60, 61, n)
df['stakeholder_issues'] = rng.randint(1, 6, n)
df['stakeholder_eng'] = rng.randint(1, 4, n)
df['stakeholder_influence'] = rng.randint(1, 4, n)
df['external_deps'] = rng.randint(1, 4, n)
df['economic_stable'] = rng.choice([0,1], n)
df['quality_issues'] = rng.choice([0,1], n)

In [ ]:
def assign_risk(row):

    if row['tech_complexity']>=4 and row['security_threats']>=4:
        risk_type = 'Technical'
    elif row['cost_variance']>200_000 or row['budget_changes']>=4:
        risk_type = 'Financial'
    elif row['reg_impact']>=4 or row['legal_challenges']>=4:
        risk_type = 'Compliance'
    else:
        risk_type = 'Operational'

    #for impact
    impact = int((row['tech_complexity'] + row['budget_changes'] +
                  row['stakeholder_influence'])/3*2 + 1)

    # for probability
    prob = int(20 + (row['scope_creep'] + row['schedule_changes'] +
                     row['deviate_strategy'])/3*16)
    #o/p
    if impact>=7 and prob>=70:
        response = 'Mitigate'
    elif impact<=4 and prob<=40:
        response = 'Accept'
    elif risk_type=='Financial':
        response = 'Transfer'
    else:
        response = 'Avoid'

    return risk_type, impact, prob, response

df[['risk_type','impact','probability','response']] = df.apply(assign_risk, axis=1, result_type='expand')

In [ ]:
cat_cols = ['industry']
enc = OrdinalEncoder()
df[cat_cols] = enc.fit_transform(df[cat_cols])

In [ ]:
X = df.drop(columns=['project_id','risk_type','impact','probability','response'])
y_type  = df['risk_type']
y_imp   = df['impact']
y_prob  = df['probability']
y_resp  = df['response']


In [ ]:
X_train, X_test, yt_tr, yt_te, yi_tr, yi_te, yp_tr, yp_te, yr_tr, yr_te = \
    train_test_split(X, y_type, y_imp, y_prob, y_resp,
                     test_size=.2, random_state=42)

In [ ]:
#random forest
rf_clf = MultiOutputClassifier(
            RandomForestClassifier(n_estimators=100, random_state=42))
rf_reg = MultiOutputRegressor(
            RandomForestRegressor(n_estimators=100, random_state=42))

In [ ]:
rf_clf.fit(X_train, pd.concat([yt_tr, yr_tr], axis=1))
rf_reg.fit(X_train, pd.concat([yi_tr, yp_tr], axis=1))

In [ ]:
#decision tree
dt_clf = MultiOutputClassifier(
            DecisionTreeClassifier(random_state=42))
dt_reg = MultiOutputRegressor(
            DecisionTreeRegressor(random_state=42))

In [ ]:


dt_clf.fit(X_train, pd.concat([yt_tr, yr_tr], axis=1))
dt_reg.fit(X_train, pd.concat([yi_tr, yp_tr], axis=1))

In [ ]:
def evaluate_classification(y_true, y_pred, labels):
    return {'accuracy' : accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
            'recall'   : recall_score(y_true, y_pred, average='weighted', zero_division=0),
            'f1'       : f1_score(y_true, y_pred, average='weighted', zero_division=0)}

def evaluate_regression(y_true, y_pred):
    return {'MSE' : mean_squared_error(y_true, y_pred),
            'RMSE': mean_squared_error(y_true, y_pred, squared=False),
            'R2'  : r2_score(y_true, y_pred),
            'MAPE': np.mean(np.abs((y_true - y_pred) / y_true)) * 100}

In [ ]:
#random forest
pred_type_rf, pred_resp_rf = rf_clf.predict(X_test).T
pred_imp_rf,  pred_prob_rf = rf_reg.predict(X_test).T


In [ ]:
#decision tree
pred_type_dt, pred_resp_dt = dt_clf.predict(X_test).T
pred_imp_dt,  pred_prob_dt = dt_reg.predict(X_test).T

In [ ]:
from sklearn.metrics import mean_absolute_error

type_classes   = sorted(yt_te.unique())
resp_classes   = sorted(yr_te.unique())

y_true_type  = np.array([type_classes.index(l)   for l   in yt_te])
y_true_resp  = np.array([resp_classes.index(l)   for l   in yr_te])


y_pred_type  = np.array([type_classes.index(l)   for l   in pred_type_rf])
y_pred_resp  = np.array([resp_classes.index(l)   for l   in pred_resp_rf])


y_true_clf  = np.column_stack([y_true_type, y_true_resp])
y_pred_clf  = np.column_stack([y_pred_type, y_pred_resp])


incorrect_labels_rf = (y_true_clf != y_pred_clf).sum()
total_labels = y_true_clf.shape[0] * y_true_clf.shape[1]
hamming   = incorrect_labels_rf / total_labels

exact_m   = (y_true_clf == y_pred_clf).all(axis=1).mean()


mae_imp  = mean_absolute_error(yi_te, pred_imp_rf)
mae_prob = mean_absolute_error(yp_te, pred_prob_rf)


print(f"Random-Forest  |  Hamming-Loss = {hamming:.4f}  "
      f"|  Exact-Match = {exact_m:.4f}  "
      f"|  MAE-impact = {mae_imp:.4f}  "
      f"|  MAE-prob = {mae_prob:.4f}")

In [ ]:
pd.DataFrame(metrics).T

In [ ]:
import joblib, os

os.makedirs("../models", exist_ok=True)

joblib.dump(enc, "../models/ordinal_encoder.joblib")
joblib.dump(rf_clf, "../models/random_forest_multioutput_classifier.joblib")
joblib.dump(rf_reg, "../models/random_forest_multioutput_regressor.joblib")
joblib.dump(dt_clf, "../models/decision_tree_multioutput_classifier.joblib")
joblib.dump(dt_reg, "../models/decision_tree_multioutput_regressor.joblib")

# Save the synthetic dataset (optional, small enough)
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/synthetic_project_risk_dataset.csv", index=False)

print("Saved models to ../models and dataset to ../data")
